# ElasticNet 回归：L1 + L2 的最佳结合
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：ElasticNet.py | 核心功能：L1/L2 混合正则化、l1_ratio 调参

## 概述

ElasticNet 结合了 Lasso 的 L1 正则化和岭回归的 L2 正则化。损失函数：||y - Xw||² + α[l1_ratio·||w||₁ + 0.5·(1-l1_ratio)·||w||²]。

它解决了两个问题：
1. 当特征高度相关时，Lasso 可能随机选一个，ElasticNet 倾向于同时选择或同时排除相关特征
2. 当特征数 > 样本数时，Lasso 最多选 n 个特征，ElasticNet 没有这个限制

脚本在含相关特征的数据上对比了四种回归方法，并展示了 l1_ratio 和 alpha 的网格搜索。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet, ElasticNetCV, Lasso, Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
# --- 导入库 ---
from sklearn.metrics import r2_score

## 数学原理

### 1. ElasticNet 的目标函数

**代码对应**：`ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_train, y_train)` 同时使用 L1 和 L2 正则化。

ElasticNet 结合了 Lasso（L1）和 Ridge（L2）的惩罚项：

$$J(\mathbf{w}) = \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha\left[\rho\|\mathbf{w}\|_1 + \frac{1-\rho}{2}\|\mathbf{w}\|_2^2\right]$$

其中 $\rho \in [0, 1]$ 为 `l1_ratio` 参数，控制 L1 和 L2 的混合比例。

等价形式（sklearn 使用的参数化）：

$$J(\mathbf{w}) = \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha_1\|\mathbf{w}\|_1 + \frac{\alpha_2}{2}\|\mathbf{w}\|_2^2$$

其中 $\alpha_1 = \alpha \rho$，$\alpha_2 = \alpha(1-\rho)$。

### 2. l1_ratio 的极端情况

**代码对应**：代码中 `for l1_ratio in [0.0, 0.1, ..., 1.0]` 展示了从纯 Ridge 到纯 Lasso 的过渡。

- $\rho = 0$（纯 Ridge）：$J = \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \frac{\alpha}{2}\|\mathbf{w}\|_2^2$
- $\rho = 1$（纯 Lasso）：$J = \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha\|\mathbf{w}\|_1$
- $0 < \rho < 1$：同时具有 L1 的稀疏性和 L2 的分组效应

### 3. ElasticNet 的独特优势：分组效应

当多个特征高度相关时：
- **Lasso** 只选择其中一个，其余置零（不稳定）
- **ElasticNet** 倾向于将相关特征**一起选择或一起排除**（分组效应，grouping effect）

**数学原因**：L2 项使得相关特征的系数趋于相等，L1 项则决定是否整体为零。相关特征之间的系数差异被 L2 惩罚压制：

$$|w_j - w_k| \leq \frac{1}{\alpha_2} \cdot \text{Corr}(\mathbf{x}_j, \mathbf{x}_k) \text{ 的某个函数}$$

### 4. 超参数选择

**代码对应**：`ElasticNetCV(l1_ratio=[...], cv=5)` 同时搜索 alpha 和 l1_ratio。

需要调优两个超参数：
- $\alpha$：总正则化强度（越大正则化越强）
- $\rho$（l1_ratio）：L1/L2 混合比例

sklearn 的 `ElasticNetCV` 使用循环坐标下降法，对每个 $(\alpha, \rho)$ 组合进行交叉验证：

$$\hat{w}_j^{(t)} \leftarrow S_{\alpha\rho/c_j}\left(\frac{\frac{1}{n}\sum_i x_{ij}r_i^{(-j)} + \alpha_2 w_j^{(t-1)}}{1 + \alpha_2/c_j}\right)$$

其中 $r_i^{(-j)} = y_i - \hat{y}_i^{(-j)}$ 为去掉第 $j$ 个特征贡献后的残差。

### 5. 正则化家族总结

| 方法 | 惩罚项 | 闭式解？ | 稀疏？ | 分组？ |
|------|--------|---------|--------|--------|
| OLS | 无 | 是 | 否 | 否 |
| Ridge | $\alpha\|\mathbf{w}\|_2^2$ | 是 | 否 | 否 |
| Lasso | $\alpha\|\mathbf{w}\|_1$ | 否 | 是 | 否 |
| ElasticNet | $\alpha[\rho\|\mathbf{w}\|_1 + \frac{1-\rho}{2}\|\mathbf{w}\|_2^2]$ | 否 | 是 | 是 |

选择指南：特征多且稀疏用 Lasso；特征间有共线性用 Ridge；两者都需要用 ElasticNet。

### 1. 构造数据

运行 1. 构造数据 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n, p = 200, 15
X = np.random.randn(n, p)
true_coef = np.zeros(p)
true_coef[:3] = [5, -3, 2]  # 只有前 3 个有用
# --- 赋值/配置 ---
true_coef[3] = 1.5
y = X @ true_coef + np.random.randn(n) * 0.5

# 添加高度相关的特征组
X[:, 10] = X[:, 0] * 0.9 + np.random.randn(n) * 0.05
X[:, 11] = X[:, 1] * 0.85 + np.random.randn(n) * 0.05

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 2. 三种回归对比

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)
en = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_train, y_train)

print("=== 四种回归对比 ===")
print(f"{'方法':>18} {'测试 R²':>10} {'非零系数':>10}")
print(f"{'线性回归':>18} {lr.score(X_test, y_test):>10.4f} {np.count_nonzero(lr.coef_):>10}")
print(f"{'Ridge (L2)':>18} {ridge.score(X_test, y_test):>10.4f} {np.count_nonzero(ridge.coef_):>10}")
print(f"{'Lasso (L1)':>18} {lasso.score(X_test, y_test):>10.4f} {np.count_nonzero(lasso.coef_):>10}")
# --- 输出结果 ---
print(f"{'ElasticNet':>18} {en.score(X_test, y_test):>10.4f} {np.count_nonzero(en.coef_):>10}")

### 3. l1_ratio 参数

探索关键超参数对模型行为的影响。

In [ ]:
# l1_ratio=1 等价于 Lasso, l1_ratio=0 等价于 Ridge
print("\n=== l1_ratio 对比（alpha=0.1）===")
for l1_ratio in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    en_r = ElasticNet(alpha=0.1, l1_ratio=l1_ratio, max_iter=10000)
    en_r.fit(X_train, y_train)
    n_nz = np.count_nonzero(en_r.coef_)
# --- 条件判断 ---
    name = "Ridge" if l1_ratio == 0 else ("Lasso" if l1_ratio == 1 else f"EN")
    print(f"  l1_ratio={l1_ratio}: 测试R²={en_r.score(X_test, y_test):.4f}, "
          f"非零系数={n_nz}/{p}")

### 4. 超参数网格搜索

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== GridSearchCV (alpha + l1_ratio) ===")
param_grid = {
    "alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
    "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
}
# --- 继续 ---
gs = GridSearchCV(
    ElasticNet(max_iter=10000, random_state=42),
    param_grid, cv=5, scoring="r2", n_jobs=-1
)
gs.fit(X_train, y_train)
# --- 输出结果 ---
print(f"最佳参数: {gs.best_params_}")
print(f"最佳交叉验证 R²: {gs.best_score_:.4f}")
print(f"测试集 R²: {gs.best_estimator_.score(X_test, y_test):.4f}")

### 5. ElasticNetCV

运行 5. ElasticNetCV 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== ElasticNetCV ===")
en_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, random_state=42, max_iter=10000)
en_cv.fit(X_train, y_train)
print(f"最佳 alpha: {en_cv.alpha_:.6f}")
print(f"最佳 l1_ratio: {en_cv.l1_ratio_}")
# --- 输出结果 ---
print(f"测试 R²: {en_cv.score(X_test, y_test):.4f}")

print("\n=== ElasticNet 要点 ===")
print("损失函数: ||y - Xw||² + α × [l1_ratio × ||w||₁ + 0.5 × (1-l1_ratio) × ||w||²]")
print("- l1_ratio=1 → 纯 Lasso (L1)，可做特征选择")
print("- l1_ratio=0 → 纯 Ridge (L2)，处理共线性")
print("- 0<l1_ratio<1 → 两者兼顾，尤其适合特征高度相关时")

## 关键代码解释

### l1_ratio 的"刻度盘"

```python
for l1_ratio in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    en_r = ElasticNet(alpha=0.1, l1_ratio=l1_ratio)
```

- l1_ratio=0 → 纯岭回归（L2）
- l1_ratio=1 → 纯 Lasso（L1）
-  < l1_ratio < 1 → 混合

通常 l1_ratio 在 [0.1, 0.9] 之间网格搜索。

### ElasticNetCV 自动双参数搜索

```python
en_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5)
```

同时优化 alpha 和 l1_ratio，比手动 GridSearchCV 更高效（内部使用热启动路径算法）。

## 使用示例

```python
from sklearn.linear_model import ElasticNetCV
en_cv = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5)
en_cv.fit(X_train, y_train)
print(f"alpha={en_cv.alpha_}, l1_ratio={en_cv.l1_ratio_}")
```

## 注意事项

1. **必须特征缩放**
2. **ElasticNetCV 比 GridSearchCV 更高效**：内部使用坐标下降的路径算法
3. **在共线性场景下优于纯 Lasso**
4. **两个超参数需要调**：比 Ridge 或 Lasso 多一个维度的搜索空间

## 总结与延伸

以上代码展示了 **ElasticNet** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Group ElasticNet**：对特征分组，组内用 L2，组间用 L1
- **稀疏组 Lasso**：结合组稀疏和特征稀疏
- **在线 ElasticNet**：SGDRegressor(penalty="elasticnet") 支持在线学习
